# C7 · Curación de alineamientos, IQ-TREE e iTOL

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/07_filogenia/07_filogenia_iqtree_itol.ipynb)

## Pregunta guía

Un árbol agrupa secuencias, pero contiene parálogos, fragmentos y soportes distintos. **¿Qué conclusión es compatible con el alineamiento y qué afirmación excede la evidencia?**

### Objetivos

- integrar resultados y metadatos con herramientas tabulares;
- justificar la inclusión/exclusión de secuencias;
- ejecutar selección de modelo y máxima verosimilitud con IQ-TREE;
- interpretar UFBoot y SH-aLRT sin convertirlos en probabilidad de que una hipótesis sea “verdadera”;
- generar anotaciones iTOL;
- expresar el flujo C6–C7 como dependencias reproducibles.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Integrar metadatos antes de interpretar

In [ ]:
import pandas as pd
from Bio import AlignIO

meta = pd.read_csv(ROOT / "data/module07/metadata.tsv", sep="\t")
aln = AlignIO.read(ROOT / "data/module07/homologs_aligned.faa", "fasta")
aln_ids = {record.id for record in aln}
meta_ids = set(meta.sequence_id)
print("En alineamiento, sin metadata:", sorted(aln_ids - meta_ids))
print("En metadata, sin alineamiento:", sorted(meta_ids - aln_ids))
meta

## 2. Curación explícita

Registre cada exclusión con criterio y evidencia. Ejemplos: cobertura insuficiente, secuencia casi idéntica redundante, contaminación probable, región no alineable o anotación inconsistente. No elimine secuencias solo porque “dañan el árbol”.

In [ ]:
import numpy as np

rows = []
for record in aln:
    seq = str(record.seq)
    rows.append({
        "sequence_id": record.id,
        "length_aligned": len(seq),
        "gap_fraction": seq.count("-") / len(seq),
        "unknown_fraction": (seq.count("X") + seq.count("?")) / len(seq),
    })
pd.DataFrame(rows)

## 3. IQ-TREE reproducible

In [ ]:
script = ROOT / "scripts/module07/run_iqtree.sh"
print(script.read_text(encoding="utf-8"))

Comando central:

```bash
iqtree2 -s alignment.faa -m MFP -B 1000 --alrt 1000 -T AUTO --prefix course_tree
```

- `-m MFP`: selección de modelo entre candidatos.
- `-B 1000`: ultrafast bootstrap.
- `--alrt 1000`: SH-aLRT.
- `-T AUTO`: selección de hilos; en HPC debe respetar `--cpus-per-task`.

Con conjuntos pequeños, la selección automática de hilos puede ser innecesaria; en Slurm, use `-T "$SLURM_CPUS_PER_TASK"`.

In [ ]:
from Bio import Phylo
from io import StringIO

tree_text = (ROOT / "data/module07/example.treefile").read_text().strip()
tree = Phylo.read(StringIO(tree_text), "newick")
Phylo.draw_ascii(tree)

### Checkpoint

Un soporte alto indica estabilidad de una bipartición bajo el procedimiento, no prueba función, dirección evolutiva ni ausencia de sesgo. La raíz requiere un outgroup o un supuesto explícito.

## 4. Archivo de anotación iTOL

In [ ]:
import subprocess
out = ROOT / "results/module07/itol_groups.txt"
out.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "python", "scripts/module07/make_itol_dataset.py",
    "data/module07/metadata.tsv", str(out)
], cwd=ROOT, check=True)
print(out.read_text())

## 5. Del notebook al workflow

In [ ]:
print((ROOT / "workflow/Snakefile").read_text(encoding="utf-8"))

Ejecute primero un *dry run*:

```bash
snakemake -s workflow/Snakefile -n -p
```

Un workflow hace explícito qué salida depende de qué entrada y evita repetir manualmente pasos fuera de orden.

## Reto

Produzca árbol, modelo, soportes, metadatos y una figura iTOL. Escriba una interpretación de máximo 500 palabras que separe observación, inferencia y limitación.